### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch
from scipy.stats import mannwhitneyu

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [4]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [5]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are tuned once using Optuna with StratifiedGroupKFold (5 splits) on the full dataset, ensuring no subject appears in both train and validation folds. Best parameters are then frozen and used for final LOSO evaluation. Results are also evaluated using 10-fold CV for direct comparison with existing literature.

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [10]:
#Stops hyperparameter tuning if no improvement after a set # of trials
def no_improvement_callback(study, trial, n_trials_no_improve=50):
    if trial.number >= n_trials_no_improve:
        recent_values = [t.value for t in study.trials[-n_trials_no_improve:]]
        if max(recent_values) <= study.best_value:
            study.stop()

# Class balancing
neg_count = np.sum(y_en == 0)
pos_count = np.sum(y_en == 1)
scale_en = neg_count / pos_count

# Tune once on full dataset with StratifiedGroupKFold
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_en,
        'eval_metric':      'logloss',
    }
    model = XGBClassifier(**params, n_jobs=-1)
    cv = StratifiedGroupKFold(n_splits=5)
    scores = cross_val_score(model, X_en, y_en, cv=cv, groups=groups_en, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, callbacks=[lambda s, t: no_improvement_callback(s, t)])
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# Save best params
best_params = study.best_params
best_params['scale_pos_weight'] = scale_en
best_params['random_state'] = 42

[I 2026-03-02 11:51:16,528] A new study created in memory with name: no-name-9bdd7355-8b65-476b-8148-37b78c99be73
[I 2026-03-02 11:51:20,176] Trial 0 finished with value: 0.46979890903519017 and parameters: {'n_estimators': 434, 'max_depth': 3, 'learning_rate': 0.01961226977410001, 'subsample': 0.8250711472145763, 'colsample_bytree': 0.9257098979211792, 'min_child_weight': 1, 'gamma': 2.7300617496042427}. Best is trial 0 with value: 0.46979890903519017.
[I 2026-03-02 11:51:27,908] Trial 1 finished with value: 0.4670280791965034 and parameters: {'n_estimators': 439, 'max_depth': 6, 'learning_rate': 0.010203603638899194, 'subsample': 0.6424745653988759, 'colsample_bytree': 0.6814910909632091, 'min_child_weight': 1, 'gamma': 0.34077258833736856}. Best is trial 0 with value: 0.46979890903519017.
[I 2026-03-02 11:51:35,005] Trial 2 finished with value: 0.4663704294670678 and parameters: {'n_estimators': 241, 'max_depth': 8, 'learning_rate': 0.019542990042286153, 'subsample': 0.7174491629062

Best params: {'n_estimators': 415, 'max_depth': 4, 'learning_rate': 0.21131862439209098, 'subsample': 0.9450115514945492, 'colsample_bytree': 0.7448590338277459, 'min_child_weight': 9, 'gamma': 0.622936831361465}
Best CV accuracy: 0.4981


In [11]:
# LOSO evaluation with fixed params
logo = LeaveOneGroupOut()
en_accuracies = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_en, y_en, groups_en)):
    X_train, X_test = X_en[train_idx], X_en[test_idx]
    y_train, y_test = y_en[train_idx], y_en[test_idx]

    model = XGBClassifier(**best_params, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    en_accuracies.append(accuracy_score(y_test, preds))
    print(f"Fold {fold+1} | Accuracy: {en_accuracies[-1]:.4f}")

print("\n=== LOSO XGB (EN) ===")
print(f"Accuracy:  {np.mean(en_accuracies):.4f} ± {np.std(en_accuracies):.4f}")


Fold 1 | Accuracy: 0.5946
Fold 2 | Accuracy: 0.4225
Fold 3 | Accuracy: 0.4939
Fold 4 | Accuracy: 0.5909
Fold 5 | Accuracy: 0.4892
Fold 6 | Accuracy: 0.5814
Fold 7 | Accuracy: 0.7579
Fold 8 | Accuracy: 0.4231
Fold 9 | Accuracy: 0.6429
Fold 10 | Accuracy: 0.4052
Fold 11 | Accuracy: 0.6432
Fold 12 | Accuracy: 0.3052
Fold 13 | Accuracy: 0.2857
Fold 14 | Accuracy: 0.4476
Fold 15 | Accuracy: 0.4933
Fold 16 | Accuracy: 0.5266
Fold 17 | Accuracy: 0.4566
Fold 18 | Accuracy: 0.4490
Fold 19 | Accuracy: 0.6739
Fold 20 | Accuracy: 0.6667
Fold 21 | Accuracy: 0.4907
Fold 22 | Accuracy: 0.4795
Fold 23 | Accuracy: 0.3613
Fold 24 | Accuracy: 0.2055
Fold 25 | Accuracy: 0.5000
Fold 26 | Accuracy: 0.4167
Fold 27 | Accuracy: 0.4909
Fold 28 | Accuracy: 0.5000
Fold 29 | Accuracy: 0.4839
Fold 30 | Accuracy: 0.6032
Fold 31 | Accuracy: 0.4882
Fold 32 | Accuracy: 0.5014
Fold 33 | Accuracy: 0.4338
Fold 34 | Accuracy: 0.4715

=== LOSO XGB (EN) ===
Accuracy:  0.4934 ± 0.1125


In [12]:
# 10 Fold Cross CV with same tuned params
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_10fold = cross_val_score(
    XGBClassifier(**best_params, n_jobs=-1),
    X_en, y_en,
    cv=cv_10fold,
    scoring='accuracy',
    n_jobs=-1
)

print("\n=== 10-FoldCV XGB (EN) ===")
print(f"Accuracy: {scores_10fold.mean():.4f} ± {scores_10fold.std():.4f}")


=== 10-FoldCV XGB (EN) ===
Accuracy: 0.6924 ± 0.0129


##### 5.1.2 Positive vs. Negative

In [13]:
#Stops hyperparameter tuning if no improvement after a set # of trials
def no_improvement_callback(study, trial, n_trials_no_improve=20):
    if trial.number >= n_trials_no_improve:
        recent_values = [t.value for t in study.trials[-n_trials_no_improve:]]
        if max(recent_values) <= study.best_value:
            study.stop()

# Class balancing
neg_count = np.sum(y_pn == 0)
pos_count = np.sum(y_pn == 1)
scale_pn = neg_count / pos_count

# Tune once on full dataset with StratifiedGroupKFold
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_pn,
        'eval_metric':      'logloss',
    }
    model = XGBClassifier(**params, n_jobs=-1)
    cv = StratifiedGroupKFold(n_splits=5)
    scores = cross_val_score(model, X_pn, y_pn, cv=cv, groups=groups_pn, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, callbacks=[lambda s, t: no_improvement_callback(s, t)])
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# Save best params
best_params = study.best_params
best_params['scale_pos_weight'] = scale_pn
best_params['random_state'] = 42

[I 2026-03-02 11:54:52,262] A new study created in memory with name: no-name-e8352971-8ec0-4370-b42a-ac91b24b48b0
[I 2026-03-02 11:54:54,839] Trial 0 finished with value: 0.5374052683969861 and parameters: {'n_estimators': 446, 'max_depth': 5, 'learning_rate': 0.08342473982388482, 'subsample': 0.7460504588515543, 'colsample_bytree': 0.7787454234871592, 'min_child_weight': 1, 'gamma': 2.7027029148075266}. Best is trial 0 with value: 0.5374052683969861.
[I 2026-03-02 11:54:55,867] Trial 1 finished with value: 0.5213855010294137 and parameters: {'n_estimators': 122, 'max_depth': 3, 'learning_rate': 0.04931109834992453, 'subsample': 0.7286531057329425, 'colsample_bytree': 0.8173955534495552, 'min_child_weight': 1, 'gamma': 3.9058261966167374}. Best is trial 0 with value: 0.5374052683969861.
[I 2026-03-02 11:54:59,025] Trial 2 finished with value: 0.531373047317182 and parameters: {'n_estimators': 343, 'max_depth': 6, 'learning_rate': 0.011321716727510783, 'subsample': 0.8686723471111844, '

Best params: {'n_estimators': 242, 'max_depth': 8, 'learning_rate': 0.010078924860162075, 'subsample': 0.6033743980978856, 'colsample_bytree': 0.7161988420830316, 'min_child_weight': 7, 'gamma': 0.10168490597860469}
Best CV accuracy: 0.5586


In [14]:
# LOSO evaluation with fixed params
logo = LeaveOneGroupOut()
pn_accuracies = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_pn, y_pn, groups_pn)):
    X_train, X_test = X_pn[train_idx], X_pn[test_idx]
    y_train, y_test = y_pn[train_idx], y_pn[test_idx]

    model = XGBClassifier(**best_params, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    pn_accuracies.append(accuracy_score(y_test, preds))
    print(f"Fold {fold+1} | Accuracy: {pn_accuracies[-1]:.4f}")

print("\n=== LOSO XGB (PN) ===")
print(f"Accuracy:  {np.mean(pn_accuracies):.4f} ± {np.std(pn_accuracies):.4f}")


Fold 1 | Accuracy: 0.5449
Fold 2 | Accuracy: 0.5519
Fold 3 | Accuracy: 0.6241
Fold 4 | Accuracy: 0.4713
Fold 5 | Accuracy: 0.5035
Fold 6 | Accuracy: 0.8333
Fold 7 | Accuracy: 0.4286
Fold 8 | Accuracy: 0.8231
Fold 9 | Accuracy: 0.7931
Fold 10 | Accuracy: 0.6944
Fold 11 | Accuracy: 0.1000
Fold 12 | Accuracy: 0.7653
Fold 13 | Accuracy: 0.5158
Fold 14 | Accuracy: 0.6100
Fold 15 | Accuracy: 0.3957
Fold 16 | Accuracy: 0.7059
Fold 17 | Accuracy: 0.4231
Fold 18 | Accuracy: 0.5054
Fold 19 | Accuracy: 0.4444
Fold 20 | Accuracy: 0.3571
Fold 21 | Accuracy: 0.8095
Fold 22 | Accuracy: 0.5828
Fold 23 | Accuracy: 0.2424
Fold 24 | Accuracy: 0.4022
Fold 25 | Accuracy: 0.6786
Fold 26 | Accuracy: 0.0000
Fold 27 | Accuracy: 0.4861
Fold 28 | Accuracy: 0.4286
Fold 29 | Accuracy: 0.4773
Fold 30 | Accuracy: 0.4583
Fold 31 | Accuracy: 0.3506

=== LOSO XGB (PN) ===
Accuracy:  0.5164 ± 0.1945


In [15]:
# 10 Fold Cross CV with same tuned params
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_10fold = cross_val_score(
    XGBClassifier(**best_params, n_jobs=-1),
    X_pn, y_pn,
    cv=cv_10fold,
    scoring='accuracy',
    n_jobs=-1
)

print("\n=== 10-FoldCV XGB (PN) ===")
print(f"Accuracy: {scores_10fold.mean():.4f} ± {scores_10fold.std():.4f}")


=== 10-FoldCV XGB (PN) ===
Accuracy: 0.6732 ± 0.0236
